In [1]:
import asyncio
from datetime import datetime, timedelta, timezone
from skyfield.api import load, EarthSatellite, wgs84
import random

class OrbitalSpigot:
    def __init__(self, start_time=None, time_dilation_factor=600):
        """
        time_dilation_factor: How fast time moves.
        600 means 1 real second = 600 simulation seconds (10 minutes).
        This is critical so your demo shows a full orbit in ~9 real-world seconds.
        """
        print("Initializing Async Orbital Physics Engine...")
        self.ts = load.timescale()
        self.eph = load('de421.bsp')

        # Dawn-Dusk SSO Proxy TLE
        line1 = '1 23710U 95059A   23301.12345678  .00000123  00000-0  12345-4 0  9999'
        line2 = '2 23710  98.5780  80.1234 0001234  90.0000 270.0000 14.29812345123456'
        self.satellite = EarthSatellite(line1, line2, 'Tsukuyomi-1', self.ts)

        # Set the starting clock
        self.current_sim_time = start_time or datetime.now(timezone.utc)
        self.time_dilation = time_dilation_factor
        
    def _calculate_state(self) -> dict:
        """Internal synchronous math function."""
        t = self.ts.from_datetime(self.current_sim_time)
        geocentric = self.satellite.at(t)
        subpoint = wgs84.subpoint(geocentric)
        
        is_sunlit = geocentric.is_sunlit(self.eph)
        debris_warning = random.random() < 0.005 # 0.5% chance per tick

        return {
            "timestamp": self.current_sim_time.isoformat(),
            "orbit_type": "SSO (Dawn-Dusk)",
            "is_sunlit": bool(is_sunlit),
            "latitude": round(subpoint.latitude.degrees, 4),
            "longitude": round(subpoint.longitude.degrees, 4),
            "altitude_km": round(subpoint.elevation.km, 2),
            "hazard_debris_warning": debris_warning
        }
    async def telemetry_stream(self, current_sim_time):
        """
        ASYNC GENERATOR: This is what FastAPI will consume.
        It yields a new state, advances the clock, and waits 1 real second.
        """
        while True:
            # 1. Calculate current physics
            state = self._calculate_state()
            
            # 2. Yield data to FastAPI
            yield state
            
            # 3. Fast-forward the simulation clock by our dilation factor
            self.current_sim_time += timedelta(seconds=self.time_dilation)
            
            # 4. Sleep for 1 real-world second before the next tick
            await asyncio.sleep(1)

In [14]:
# ==========================================
# LOCAL TERMINAL TESTER
# Run this file directly (python orbital_model.py) to test the async generator
# without needing the FastAPI server.
# ==========================================
if __name__ == "__main__":
    async def run_test():
        print("🚀 Starting PHOENIX Orbital Spigot Test...")
        print("Time Dilation: 1 real second = 10 simulation minutes")
        print("-" * 50)
        
        # 1. Initialize the spigot
        spigot = OrbitalSpigot(time_dilation_factor=600)
        
        # 2. Create the async generator
        stream = spigot.telemetry_stream()
        
        # 3. Pull the first 10 ticks to verify it works
        try:
            for _ in range(10):
                # Fetch the next yielded state from the generator
                state = await anext(stream)
                
                # Format for the terminal
                sun_status = "☀️ SUNLIT (Charging)" if state["is_sunlit"] else "🌑 ECLIPSE (Draining)"
                debris_status = "⚠️ DEBRIS WARNING" if state["hazard_debris_warning"] else "✅ CLEAR"
                
                print(f"Time: {state['timestamp']}")
                print(f"Position: Lat {state['latitude']}, Lon {state['longitude']}, Alt {state['altitude_km']} km")
                print(f"Illumination: {sun_status}")
                print(f"Radar: {debris_status}")
                print("-" * 50)
                
        except KeyboardInterrupt:
            print("Test stopped manually.")

    # Execute the async test
    asyncio.run(run_test())

NameError: name 'telemetry_stream' is not defined